### Sentences to token id's

- A sentence is broken down into words and punctuations, with white space removed(We might need to keep white space depending on the application for example python code). 
- These words and puntuations are used to make a vocabulary(/dictonary), where each word is given a number(mapped to a number). These numbers are the token id's of their coresponding words.
- When we exchange the words list with their token id's we get token list.(own terms are being used)
- Some special characters(simply speaking) to represent end of the document and unknown words with their own token id's usually at the end of the dictonary(highest token values).

In [4]:
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/"
"LLMs-from-scratch/main/ch02/01_main-chapter-code/"
"the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x114b89e50>)

In [5]:
with open('the-verdict.txt', "r", encoding="utf-8") as f:
    raw_text = f.read()
print(len(raw_text))
print(raw_text[:100])

20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [6]:
import re
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [word.strip() for word in preprocessed if word.strip()] 
                             # empty string is considered as False
print(len(preprocessed))

4690


In [7]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [8]:
all_words = sorted(set(preprocessed))
len(all_words)

1130

In [9]:
vocab = {token: integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 15:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)


In [10]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = ' '.join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [11]:
tokenizer = SimpleTokenizerV1(vocab)
text = """ It's the last he painted, you know,"
    Mrs. Gisburn said with pardonable pride.
"""
ids = tokenizer.encode(text)
ids

[56,
 2,
 850,
 988,
 602,
 533,
 746,
 5,
 1126,
 596,
 5,
 1,
 67,
 7,
 38,
 851,
 1108,
 754,
 793,
 7]

In [12]:
tokenizer.decode(ids)

'It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [13]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: integer for integer, token in enumerate(all_tokens)}
len(vocab.items())

1132

In [14]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [15]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {integer:token for token, integer in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int
                        else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    def decode(self, ids):
        text = " ".join([self.int_to_str[s] for s in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text
        

In [16]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
text

'Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.'

In [17]:
tokenizer = SimpleTokenizerV2(vocab)
x = tokenizer.encode(text)

In [18]:
tokenizer.decode(x)

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

### Byte pair encoding

- BPE is an algorithm used for encoding.
- It can handle words that are not part of our vocabulary
- It breaks down the words into individual characters.
- Then it counts the number of pairs that are commonly seen(highest frequency). It combines and makes the pair a single token and repeats the stratagy.

In [19]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [20]:
text = (
"Hello, do you like tea? <|endoftext|> In the sunlit terraces "
"of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
integers

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 554,
 262,
 4252,
 18250,
 8812,
 2114,
 286,
 617,
 34680,
 27271,
 13]

In [21]:
strings = tokenizer.decode(integers)
strings

'Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.'

Exercise 2.1

In [44]:
n = "Akwirw ier"
d = tokenizer.encode(n)
print(d)
print(tokenizer.decode([33901]), tokenizer.decode([86]), tokenizer.decode([343]))
tokenizer.decode(d)

[33901, 86, 343, 86, 220, 959]
Ak w ir


'Akwirw ier'

In [28]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [29]:
enc_text = tokenizer.encode(raw_text)
len(enc_text)

5145

### Data sampling with a sliding window
input-target pairs

In [30]:
enc_sample = enc_text[50:]

In [32]:
context_size = 4
X = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print(f"x: {X}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [33]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "->" ,desired)


[290] -> 4920
[290, 4920] -> 2241
[290, 4920, 2241] -> 287
[290, 4920, 2241, 287] -> 257


In [37]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "-->", tokenizer.decode([desired]))

 and -->  established
 and established -->  himself
 and established himself -->  in
 and established himself in -->  a


Using pytorch's custom dataset to store the input-target pairs

In [47]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        """
        - txt is the text we want to tokenize
        - tokenizer is an instance of tokenizer
        - max_length is the length of each training sequence/input
        - stride how much shift in creating the next sample
        """
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chuck = token_ids[i:i+max_length]
            target_chuck = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chuck))
            self.target_ids.append(torch.tensor(target_chuck))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]


- Dataset only stores our data and returns one sample at a time.
- Dataloader helps us get data in batches(32 inputs at time, etc)
- It can shuffles data for us
- Parallely load data using multiple cpu processes
- Automatic iteration

In [48]:
def create_dataloader_v1(txt, batch_size=4, max_length=255, stride=128, shuffle=True, drop_last=True, nums_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length=max_length, stride=stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=nums_workers
    )
    return dataloader


In [50]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_txt = f.read()

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
second_batch = next(data_iter)
print(first_batch)
print(second_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


Excersise 2.2

In [52]:
dataloader_ex_1 = create_dataloader_v1(raw_text, batch_size=1, max_length=2, stride=2)
data_iter_ex = iter(dataloader_ex_1)
first_batch = next(data_iter_ex)
second_batch = next(data_iter_ex)
print(first_batch)
print(second_batch)

[tensor([[4808,  321]]), tensor([[321,  62]])]
[tensor([[ 338, 1243]]), tensor([[1243,   13]])]


Batch size is a hyper paramer
- Smaller batch size saves us memory but it will lead to more noisy model updates
- Next we will put stride size as 4 to utilize the dataset fully and to avoid any overlap between batches as it can lead to overfitting

In [53]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4,shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(inputs)
print(targets)

tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


### Creating embedding tokens

In [55]:
input_ids = torch.tensor([2,3,5,1])
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [56]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


- The number of rows is same as the vocab size. Each row representing the token ids vector. 
- The number of columns is the output dim size. More dim more complex relations(language is incredibely complex) are captured.
- Initially these are generated randomly and then during training these are adjusted.
- In `print(embedding_layer(torch.tensor([3])))` we are getting the embedding of token id 3. 

In [57]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


- When we do the embedding we get the same vectore for a token id irrespective of the position of the token.
- But in language position of a word also has a meaning to it.
- So we add additional info to the vectors.
- There are two types: relative positional embeddings and absolute positional embeddings. 
------------------------------------------------------------------
- Absolute positional embeddings are directly associated with specific positions in a sequence.
- A seperate embedding will be created for positions(positions will be the tokens here).
- Final input token will be the sum of position embedding and token embedding.
------------------------------------------------------------------
- In relative positional embedding instead of focusing on the absolute position of a token, the emphasis of relative posi-
tional embeddings is on the relative position or distance between tokens. This means the model learns the relationships in terms of “how far apart” rather than “at which exact position.”

Here we are only creating absolute position embedding

In [59]:
# token embedding

vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs: \n", inputs)
print("Input shape: ", inputs.shape)

Token IDs: 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Input shape:  torch.Size([8, 4])


In [62]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [63]:
# positional embeddings

context_size = max_length
pos_embedding_layer = torch.nn.Embedding(context_size, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_size))
print(pos_embeddings.shape)

torch.Size([4, 256])


Each position now has a 256 dim vector

In [64]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


pytorch will add the 4*256 with all 8 things in the batch.